In [1]:
import numpy as np
import pandas as pd
import seaborn as sns


from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_absolute_error
from nltk.cluster.kmeans import KMeansClusterer
from nltk.cluster.util import euclidean_distance
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from ucimlrepo import fetch_ucirepo
# fetch dataset
support2 = fetch_ucirepo(id=880)

import umap.umap_ as umap
import matplotlib.pyplot as plt

from models import *
from my_encodings import *

In [2]:
# --------------------------- Initialize dataset ------------------------------
# data (as pandas dataframes)
features = support2.data.features
targets = support2.data.targets


# join together as 1 df for EDA 
df = features.join(targets)
df = df.drop(columns=['death', 'hospdead', 'adlp', 'adls', 'scoma', 'totmcst', 'totcst', 'sps', 'aps', 'hday', 'adlsc', 'prg2m', 'prg6m', 'charges', 'dzgroup', 'dnr', 'dnrday', 'urine']) # drop other target features


# drop rows where target = NaN
print(f'Before dropping NaNs in target: {df.shape[0]}')
df = df.dropna(subset=['sfdm2'])

print(f'After dropping NaNs in target: {df.shape[0]}')

# check remaining columns
print(df.columns)
print(len(df.columns))

Before dropping NaNs in target: 9105
After dropping NaNs in target: 7705
Index(['age', 'sex', 'dzclass', 'num.co', 'edu', 'income', 'avtisst', 'race',
       'surv2m', 'surv6m', 'diabetes', 'dementia', 'ca', 'meanbp', 'wblc',
       'hrt', 'resp', 'temp', 'pafi', 'alb', 'bili', 'crea', 'sod', 'ph',
       'glucose', 'bun', 'sfdm2'],
      dtype='object')
27


Pipeline for encoding, scaling, and imputing NaNs

https://www.geeksforgeeks.org/machine-learning/handling-missing-data-with-knn-imputer/

for numerical features
   1) impute
   2) scale


for categorical features
   1) encode to numerical
   2) impute

In [3]:
X = df.drop(columns=['sfdm2'])
y = df[['sfdm2']]

# split into training vs test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

# # using LabelEncoder
# le = LabelEncoder()
# y_encoded = le.fit_transform(y_train)
# y_train = pd.DataFrame(y_encoded, columns=['sfdm2'])

# y_test_encoded = le.transform(y_test)
# y_test = pd.DataFrame(y_test_encoded, columns=['sfdm2'])
# print(f"Contains missing values: {y_train.isna().any().any()}")
# print(f"Contains missing values: {y_test.isna().any().any()}")

In [4]:
X_train.shape

(6164, 26)

In [5]:
y_train

,sfdm2
6549,SIP>=30
1659,no(M2 and SIP pres)
7399,<2 mo. follow-up
3668,adl>=4 (>=5 if sur)
2034,<2 mo. follow-up
...,...
6258,adl>=4 (>=5 if sur)
6432,no(M2 and SIP pres)
1155,<2 mo. follow-up
8986,no(M2 and SIP pres)


In [6]:
# ----------------------------- process numerical features -----------------------------
def num_feature_pre_processing(training_set: pd.DataFrame, test_set: pd.DataFrame):
    numerical_feats = training_set.select_dtypes(include=['number']).columns.tolist()
    numerical_df_train = training_set[numerical_feats]
    numerical_df_test = test_set[numerical_feats]

    # impute
    imputer = KNNImputer(n_neighbors=2) # initialize imputer 

    # training
    x_train = imputer.fit_transform(numerical_df_train) # fit and transform training set 

    # testing
    x_test = imputer.transform(numerical_df_test)

    # scale
    scaler = StandardScaler()

    # training
    x_train_scaled = scaler.fit_transform(x_train)
    x_train = pd.DataFrame(x_train, columns=numerical_df_train.columns, index=numerical_df_train.index)

    # testing
    x_test_scaled = scaler.transform(x_test)
    x_test = pd.DataFrame(x_test, columns=numerical_df_test.columns, index=numerical_df_test.index)

    # add to categorical data
    training_set[numerical_feats] = x_train[numerical_feats]
    test_set[numerical_feats] = x_test[numerical_feats]

    return training_set, test_set


In [7]:
XS_train, XS_test = num_feature_pre_processing(X_train, X_test)

In [8]:
XS_train.shape

(6164, 26)

In [9]:
# ------------------------------ process categorical features ---------------------------------
def cat_pre_processing(training_set: pd.DataFrame, test_set: pd.DataFrame, y_train, y_test):
    training_set = training_set.copy()
    test_set = test_set.copy()
    cat_cols = ['sex', 'dzclass', 'race', 'ca', 'income']
    categorical_df_train = training_set[cat_cols]
    categorical_df_test = test_set[cat_cols]

    # encode
    x_train_encoded, x_test_encoded = dummy_encode(categorical_df_train, categorical_df_test, ['sex', 'dzclass', 'race', 'ca'])

    # income
    income_order = ['under $11k', '$11-$25k', '$25-$50k', '>$50k']
    x_train_encoded, x_test_encoded = ordinal_encode(x_train_encoded, x_test_encoded, 'income', category_order=income_order)

    # target
    sfdm2_order = ['no(M2 and SIP pres)', 'adl>=4 (>=5 if sur)', 'SIP>=30', 'Coma or Intub',  '<2 mo. follow-up']
    y_train_encoded, y_test_encoded = ordinal_encode(y_train, y_test, 'sfdm2', category_order=sfdm2_order)

    # impute
    imputer = KNNImputer(n_neighbors=2) # initialize imputer 

    # training
    x_train = imputer.fit_transform(x_train_encoded) # fit and transform training set 
    x_train = pd.DataFrame(x_train, columns=x_train_encoded.columns, index=x_train_encoded.index)
    # x_train = pd.DataFrame(x_train, columns=categorical_df_train.columns, index=categorical_df_train.index)

    # testing
    x_test = imputer.transform(x_test_encoded)
    x_test = pd.DataFrame(x_test, columns=x_test_encoded.columns, index=x_test_encoded.index)
    # x_test = pd.DataFrame(x_test, columns=categorical_df_test.columns, index=categorical_df_test.index)

    x_train_num = training_set.drop(columns=cat_cols) # get numerical columns back
    x_train = pd.concat([x_train_num, x_train], axis=1)

    x_test_num = test_set.drop(columns=cat_cols) # get numerical columns back
    x_test = pd.concat([x_test_num, x_test], axis=1)

    return x_train, x_test, y_train, y_test

In [10]:
XS_train, XS_test, y_train, y_test = cat_pre_processing(XS_train, XS_test, y_train, y_test)

In [11]:
print(f"Contains missing values: {XS_train.isna().any().any()}")
print(f"Contains missing values: {XS_test.isna().any().any()}")

Contains missing values: False
Contains missing values: False


In [12]:
print(f"Contains missing values: {y_train.isna().any().any()}")
print(f"Contains missing values: {y_test.isna().any().any()}")

Contains missing values: False
Contains missing values: False


,sfdm2
6549,SIP>=30
1659,no(M2 and SIP pres)
7399,<2 mo. follow-up
3668,adl>=4 (>=5 if sur)
2034,<2 mo. follow-up
...,...
6258,adl>=4 (>=5 if sur)
6432,no(M2 and SIP pres)
1155,<2 mo. follow-up
8986,no(M2 and SIP pres)
